# Sales ETL: CSV → pandas → SQL Server

This notebook walks through a moderate ETL (**Extract, Transform, Load**) pipeline.
The fictional dataset contains **10,100 rows**: 10,000 single-product orders and
100 duplicate copies. Prices are in **EGP**, and order dates cover 2025.

Run the cells from top to bottom. You can inspect the DataFrame after each step.
To start again, restart the kernel and run all cells. Keep `sales_raw.csv` beside
this notebook. Complete the pandas steps first, then configure SQL Server before
running the database cells at the end.

## 1. Install packages

If the packages are missing, run the commented command in a terminal after removing
the `#`. Restart the notebook kernel afterward if necessary. SQL loading also requires
Microsoft ODBC Driver 18 and an existing SQL Server database.

In [1]:

import sys
print(sys.executable)
print(sys.prefix)

c:\Users\Moham\AppData\Local\Python\pythoncore-3.14-64\python.exe
c:\Users\Moham\AppData\Local\Python\pythoncore-3.14-64


## 2. Import libraries and define the output columns

`pandas` handles tabular data, and `Decimal` prepares amounts for SQL `DECIMAL` columns. `COLUMNS`
defines the exact output and insertion order.

In [3]:
import pyodbc
import pandas as pd
from decimal import Decimal

COLUMNS = ["order_id", "order_date", "customer_id", "city", "region", "product",
            "category", "quantity", "unit_price", "discount_pct", "gross_amount",
            "net_amount", "order_month", "order_size"]

ModuleNotFoundError: No module named 'pyodbc'

## 3. Read the CSV and normalize column names

`read_csv(dtype='string')` preserves dirty values for explicit conversion later. The column-name chain strips spaces, uses lowercase, and replaces spaces with underscores, so `Order ID` becomes `order_id`.

In [3]:
df = pd.read_csv('sales_raw.csv', dtype="string")
raw_count = len(df)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

df.head()

,order_id,order_date,customer_id,city,product,category,status,quantity,unit_price,discount_pct
0,100001,31/02/2026,CUST00001,,laptop,electronics,completed,1,25650,0
1,100002,14/08/2025,<NA>,Giza,Smartphone,Electronics,Completed,2,13195,5
2,100003,27/03/2025,CUST00003,Alexandria,Wireless Mouse,Accessories,Completed,0,414,10
3,100004,12/10/2025,CUST00004,Mansoura,Keyboard,Accessories,Completed,4,-25,15
4,100005,25/05/2025,CUST00005,Tanta,Office Chair,Furniture,Completed,5,4888,150


## 4. Remove exact duplicates

`drop_duplicates()` removes rows identical across all columns. We count them for reconciliation. Conflicting rows with the same order ID are checked later instead of choosing one arbitrarily.

In [4]:
df = df.drop_duplicates()
duplicate_count = raw_count - len(df)

print(f"Removed {duplicate_count:,} duplicates")

Removed 300 duplicates


## 5. Clean text and fill selected missing values

`.str.strip()` removes surrounding whitespace and `.str.title()` standardizes case. Missing cities become `Unknown`. Missing discounts become zero, an explicit assumption for this practice dataset; missing prices will not be estimated.

In [5]:
for column in ["city", "product", "category", "status"]:
    df[column] = df[column].str.strip().str.title()

df["city"] = df["city"].fillna("Unknown")

df["discount_pct"] = df["discount_pct"].fillna("0")

df[["city", "category", "status", "discount_pct"]].head()

,city,category,status,discount_pct
0,,Electronics,Completed,0
1,Giza,Electronics,Completed,5
2,Alexandria,Accessories,Completed,10
3,Mansoura,Accessories,Completed,15
4,Tanta,Furniture,Completed,150


## 6. Convert dates and numbers

`to_datetime` parses the expected year-month-day format. Commas are removed from prices before `to_numeric`. With `errors='coerce'`, invalid text becomes a missing value that validation can detect.

In [6]:
df["order_date"] = pd.to_datetime(df["order_date"], format="%d/%m/%Y", errors="coerce")
df["unit_price"] = df["unit_price"].str.replace(",", "", regex=True)
for column in ["order_id", "quantity", "unit_price", "discount_pct"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df.dtypes

order_id                 Int64
order_date      datetime64[us]
customer_id             string
city                    string
product                 string
category                string
status                  string
quantity                 Int64
unit_price             Float64
discount_pct             Int64
dtype: object

## 7. Validate and separate rejected rows

A Boolean mask checks required dates/customer IDs, positive integer order IDs and quantities, price bounds, and discounts between 0 and 100. `~valid` selects failed rows. Rejected values are already parsed; the raw CSV retains their original text. Duplicate IDs with conflicting data stop the pipeline.

In [7]:
valid = (df["order_date"].notna()
        & df["customer_id"].notna()
        & df["order_id"].between(1, 2147483647) & df["order_id"].mod(1).eq(0)
        & df["quantity"].between(1, 10000) & df["quantity"].mod(1).eq(0)
        & df["unit_price"].between(0.01, 1000000)
        & df["discount_pct"].between(0, 100))

valid = valid.fillna(False)
rejected = df.loc[~valid]
df = df.loc[valid]

if df["order_id"].duplicated().any():
    raise ValueError("Conflicting order IDs found. Resolve them before loading.")

print(f"Rejected {len(rejected):,} rows")
rejected.head()

Rejected 250 rows


,order_id,order_date,customer_id,city,product,category,status,quantity,unit_price,discount_pct
0,100001,NaT,CUST00001,,Laptop,Electronics,Completed,1,25650.0,0
1,100002,2025-08-14,<NA>,Giza,Smartphone,Electronics,Completed,2,13195.0,5
2,100003,2025-03-27,CUST00003,Alexandria,Wireless Mouse,Accessories,Completed,0,414.0,10
3,100004,2025-10-12,CUST00004,Mansoura,Keyboard,Accessories,Completed,4,-25.0,15
4,100005,2025-05-25,CUST00005,Tanta,Office Chair,Furniture,Completed,5,4888.0,150


## 8. Keep completed sales

This example loads completed orders only, excluding cancelled and returned orders. The quantity and ID columns are now safe to convert to integer types.

In [8]:
excluded_count = df["status"].ne("Completed").sum()
df = df.loc[df["status"].eq("Completed")].copy()
df[["order_id", "quantity"]] = df[["order_id", "quantity"]].astype("int64")

print(f"Excluded {excluded_count:,} non-completed orders")

df.head()

Excluded 750 non-completed orders


,order_id,order_date,customer_id,city,product,category,status,quantity,unit_price,discount_pct
1000,101001,2026-05-09,CUST01001,Cairo,Laptop,Electronics,Completed,5,29355.0,0
1001,101002,2026-12-22,CUST01002,Giza,Smartphone,Electronics,Completed,6,15080.0,5
1002,101003,2026-07-07,CUST01003,Alexandria,Wireless Mouse,Accessories,Completed,7,472.5,10
1003,101004,2026-02-20,CUST01004,Mansoura,Keyboard,Accessories,Completed,8,901.0,15
1004,101005,2026-09-05,CUST01005,Tanta,Office Chair,Furniture,Completed,9,5564.0,20


## 9. Enrich the data and calculate amounts

`map` assigns regions from cities. Gross amount = quantity × unit price. Net amount = gross amount × (1 − discount / 100), rounded to two decimals. `.dt.strftime` derives the month. `pd.cut` creates Small (≤ 1,000 EGP), Medium (> 1,000 to 10,000), and Large (> 10,000) bands. Finally, select the SQL columns and sort by ID.

In [9]:
df["region"] = df["city"].map({"Cairo": "Greater Cairo", "Giza": "Greater Cairo","Alexandria": "Coastal", "Mansoura": "Delta"}).fillna("Unknown")

df["gross_amount"] = (df["quantity"] * df["unit_price"]).round(2)

df["net_amount"] = (df["gross_amount"] * (1 - df["discount_pct"] / 100)).round(2)
df["order_month"] = df["order_date"].dt.strftime("%Y-%m")
df["order_size"] = pd.cut(df["net_amount"], bins=[-1, 1000, 10000, float("inf")],
                        labels=["Small", "Medium", "Large"]).astype("string")
clean = df[COLUMNS].sort_values("order_id").reset_index(drop=True)

clean.head()

,order_id,order_date,customer_id,city,region,product,category,quantity,unit_price,discount_pct,gross_amount,net_amount,order_month,order_size
0,101001,2026-05-09,CUST01001,Cairo,Greater Cairo,Laptop,Electronics,5,29355.0,0,146775.0,146775.0,2026-05,Large
1,101002,2026-12-22,CUST01002,Giza,Greater Cairo,Smartphone,Electronics,6,15080.0,5,90480.0,85956.0,2026-12,Large
2,101003,2026-07-07,CUST01003,Alexandria,Coastal,Wireless Mouse,Accessories,7,472.5,10,3307.5,2976.75,2026-07,Medium
3,101004,2026-02-20,CUST01004,Mansoura,Delta,Keyboard,Accessories,8,901.0,15,7208.0,6126.8,2026-02,Medium
4,101005,2026-09-05,CUST01005,Tanta,Unknown,Office Chair,Furniture,9,5564.0,20,50076.0,40060.8,2026-09,Large


## 10. Summarize sales by month and region

`groupby` forms month/region groups. Named aggregations count orders and sum units and net revenue. This summary is saved separately; the SQL table holds order-level data.

In [10]:
summary = clean.groupby(["order_month", "region"], as_index=False).agg(orders=("order_id", "count"),
                                                                        units=("quantity", "sum"),
                                                                        revenue_egp=("net_amount","sum"))
summary["revenue_egp"] = summary["revenue_egp"].round(2)
print(f"Raw: {raw_count:,}; duplicates: {duplicate_count:,}; invalid: {len(rejected):,}; "
    f"non-completed: {excluded_count:,}; clean: {len(clean):,}")

summary.head()

Raw: 10,000; duplicates: 300; invalid: 250; non-completed: 750; clean: 8,700


,order_month,region,orders,units,revenue_egp
0,2025-01,Greater Cairo,245,245,2147171.4
1,2025-01,Unknown,251,251,2160190.8
2,2025-02,Delta,237,1896,11594045.28
3,2025-02,Unknown,243,1944,11737452.92
4,2025-03,Coastal,245,735,5880045.75


## 11. Check totals and save outputs

The row counts must reconcile to the input. Order IDs must be unique, required
output values must be present, and summary revenue must match the order details.
These checks help catch accidental data loss. CSV files are written without the
DataFrame index. Running this cell again replaces these generated output files.

In [11]:
clean[clean.isna().any(axis=1)]

,order_id,order_date,customer_id,city,region,product,category,quantity,unit_price,discount_pct,gross_amount,net_amount,order_month,order_size


In [17]:
assert raw_count == duplicate_count + len(rejected) + excluded_count + len(clean)
assert clean["order_id"].is_unique
assert not clean.isna().any(axis=1).any(axis=0)
assert abs(clean["net_amount"].sum() - summary["revenue_egp"].sum()) < 0.01

clean.to_csv("sales_clean.csv", index=False)
rejected.to_csv("sales_rejected.csv", index=False)
summary.to_csv("monthly_sales_summary.csv", index=False)
print(f"Checks passed. Saved {len(clean):,} clean orders.")

Checks passed. Saved 8,700 clean orders.


## 12. Configure SQL Server

Use the server name and database you use in SQL Server Management Studio.
The database must already exist and your login needs permission to create the
table and insert/select rows. Windows Authentication uses your Windows account.

The next cell uses Windows Authentication and is configured for server `El-Neshwy`
and database `Test`. Change these values if your SSMS connection uses different names.
For SQL authentication, replace `Trusted_Connection=yes;` with `UID=...;PWD=...;`.
Do not save real passwords in a notebook that will be shared.

Keep `Encrypt=yes;TrustServerCertificate=no;` for a trusted server certificate.
For this local practice server, the code uses `TrustServerCertificate=yes`. Run the
following SQL cells only after confirming that the connection details are correct.

In [18]:
connection_string = (
    r"DRIVER={ODBC Driver 18 for SQL Server};"
    r"SERVER=localhost;"
    r"DATABASE=Test;"
    r"Trusted_Connection=yes;"
    r"Encrypt=yes;"
    r"TrustServerCertificate=yes;"
)

## 13. Define the SQL table

This cell stores SQL text; it does not connect yet. `IF OBJECT_ID ... IS NULL`
creates the table only when absent. The primary key enforces unique order IDs.
`DATE` stores dates, `NVARCHAR` stores text, and `DECIMAL` stores fixed-scale amounts.
`NOT NULL` requires values and `CHECK` constraints enforce selected rules.
The column order matches `COLUMNS`.

In [19]:
CREATE_TABLE = """
IF OBJECT_ID(N'dbo.SalesOrdersPractice', N'U') IS NULL
BEGIN
    CREATE TABLE dbo.SalesOrdersPractice (
        order_id INT NOT NULL PRIMARY KEY,
        order_date DATE NOT NULL,
        customer_id NVARCHAR(20) NOT NULL,
        city NVARCHAR(50) NOT NULL,
        region NVARCHAR(50) NOT NULL,
        product NVARCHAR(100) NOT NULL,
        category NVARCHAR(50) NOT NULL,
        quantity INT NOT NULL CHECK (quantity > 0),
        unit_price DECIMAL(12,2) NOT NULL,
        discount_pct DECIMAL(5,2) NOT NULL CHECK (discount_pct BETWEEN 0 AND 100),
        gross_amount DECIMAL(18,2) NOT NULL,
        net_amount DECIMAL(18,2) NOT NULL,
        order_month CHAR(7) NOT NULL,
        order_size NVARCHAR(10) NOT NULL
    );
END;
"""

## 14. Connect and create the SQL table

The next cell opens a SQL Server connection, creates a cursor, and executes the
`CREATE_TABLE` statement. Because that SQL statement checks whether the table already
exists, rerunning the cell will not recreate it. The transaction is then committed,
and the connection is closed.

In [20]:
connection = pyodbc.connect(connection_string)

cursor = connection.cursor()

cursor.execute(CREATE_TABLE)

connection.commit()
connection.close()

## 15. Insert the cleaned sales data

The next cell reconnects to SQL Server and defines a parameterized `INSERT` statement.
Each pandas timestamp is converted to a Python date, while monetary values are converted
to two-decimal `Decimal` values that match the SQL table definitions. The rows are inserted
in batches of 1,000 with `executemany`. Finally, the cell counts the rows in the target
table, commits the transaction, prints the total table count, and closes the connection.

> **Important:** The table uses `order_id` as its primary key. Rerunning this cell with the
> same data will cause a duplicate-key error unless the existing rows are removed or the
> loading logic is changed to an update/insert approach.

In [ ]:
connection = pyodbc.connect(connection_string)

cursor = connection.cursor()
    
insert_sql = """
        INSERT INTO dbo.SalesOrdersPractice
        (order_id, order_date, customer_id, city, region, product, category,
        quantity, unit_price, discount_pct, gross_amount, net_amount, order_month, order_size)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """

rows = []
for row in clean.itertuples(index=False, name=None):
    values = list(row)
    values[1] = values[1].date()

    for index in [8, 9, 10, 11]:
        values[index] = Decimal(str(values[index])).quantize(Decimal("0.01"))
    rows.append(tuple(values))
        
for start in range(0, len(rows), 1000):
    cursor.executemany(insert_sql, rows[start:start + 1000])

count = cursor.execute("SELECT COUNT(*) FROM dbo.SalesOrdersPractice").fetchone()[0]

connection.commit()
print(f"Committed {count:,} rows to dbo.SalesOrdersPractice.")

connection.close()